# Run Produksi — Verifikasi Realisasi Indikator RPJMN 2025-2029

**292 baris x 1 model.**

| | |
|---|---|
| Model | `watsonx-qwen3-30b-a3b-instruct-2507` |
| Backend | Serper (akurasi tertinggi di smoke test) |
| Baris | 292 (seluruh gold standard) |
| Temperature | **0** — deterministik, hasil bisa direproduksi |

## 1. Setup

In [ ]:
import os, re, json, time, hashlib, warnings, platform
from pathlib import Path
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
load_dotenv()

API_KEY  = os.getenv("MODELHUB_LLM_API_KEY")
BASE_URL = os.getenv("MODELHUB_LLM_URL", "https://api-modelhub.aiplayground.id").rstrip("/")
if not BASE_URL.endswith("/v1"):
    BASE_URL = f"{BASE_URL}/v1"
if not API_KEY:
    raise RuntimeError("MODELHUB_LLM_API_KEY belum diset di .env")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# KONFIGURASI RUN
MODEL       = "watsonx-qwen3-30b-a3b-instruct-2507"
BACKEND     = "serper"
TEMPERATURE = 0.0
MAX_TOKENS  = 800
MAX_ROUNDS  = 6

MODEL_SLUG = re.sub(r"[^a-z0-9]+", "-", MODEL.lower()).strip("-")[:32]

DATA_RAW  = Path("../data/raw")
DATA_OUT  = Path("../data/output")
CACHE_DIR = Path("../data/cache")
for d in (DATA_OUT, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

RUN_TS   = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TAG  = f"{MODEL_SLUG}_{BACKEND}"
F_JSONL  = DATA_OUT / f"raw_{RUN_TAG}.jsonl"
F_DETAIL = DATA_OUT / f"detail_{RUN_TAG}_{RUN_TS}.csv"

print("Base URL   :", BASE_URL)
print("Model      :", MODEL)
print("Backend    :", BACKEND, "| key:", "ADA" if os.getenv("SERPER_API_KEY") else "TIDAK ADA")
print("Temperature:", TEMPERATURE)
print("JSONL      :", F_JSONL)
print("Run ID     :", RUN_TS)

Base URL   : https://api-modelhub.aiplayground.id/v1
Model      : watsonx-qwen3-30b-a3b-instruct-2507
Backend    : serper | key: ADA
Temperature: 0.0
JSONL      : ..\data\output\raw_watsonx-qwen3-30b-a3b-instruct-2_serper.jsonl
Run ID     : 20260917_150324


## 2. Data (seluruh 292 baris)

In [ ]:
CSV_PATH = DATA_RAW / "gold_standard_sample.csv"
gold = pd.read_csv(CSV_PATH)
gold.columns = [c.strip() for c in gold.columns]
gold = gold.loc[:, ~gold.columns.str.startswith("Unnamed")]

COL_REAL = "realisasi (TRUE/FALSE)"
gold["_gold"] = gold[COL_REAL].astype(str).str.strip().str.upper().eq("TRUE")

# Baris yang label gold-nya diragukan -> perlu adjudikasi manual oleh tim analisis (kalau memang perlu)
GOLD_SUSPECT = [11]
gold["_gold_suspect"] = gold["no"].isin(GOLD_SUSPECT)

TARGET = gold.reset_index(drop=True)

print(f"Total baris   : {len(TARGET)}")
print(f"TRUE / FALSE  : {TARGET['_gold'].sum()} / {(~TARGET['_gold']).sum()}"
      f"  ({TARGET['_gold'].mean():.1%} TRUE)")
print(f"Gold suspect  : {TARGET['_gold_suspect'].sum()} baris -> no={GOLD_SUSPECT}")
print(f"\nBaseline 'asal TRUE' = {TARGET['_gold'].mean():.1%} akurasi, recall_FALSE = 0%")
print("\nSektor:")
print(TARGET["sektor"].value_counts().to_string())

Total baris   : 292
TRUE / FALSE  : 286 / 6  (97.9% TRUE)
Gold suspect  : 1 baris -> no=[11]

Baseline 'asal TRUE' = 97.9% akurasi, recall_FALSE = 0%

Sektor:
sektor
Sosial           61
Industri         50
Infrastruktur    38
Birokrasi        38
Kesehatan        36
Pendidikan       25
Pangan           21
Pertahanan       15
Energi            8


## 3. Search backend

In [ ]:
SEARCH_COUNTER = {"serper": 0, "tavily": 0, "cache_hit": 0}

def _cache_path(backend):
    return CACHE_DIR / f"search_cache_{backend}.json"

def _load_cache(backend):
    p = _cache_path(backend)
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

_CACHE = {"serper": _load_cache("serper"), "tavily": _load_cache("tavily")}

def _save_cache(backend):
    _cache_path(backend).write_text(
        json.dumps(_CACHE[backend], ensure_ascii=False), encoding="utf-8")

def _key(query, situs):
    return hashlib.md5(f"{query}||{situs or ''}".encode()).hexdigest()


def _serper(query, n=4):
    import requests
    api_key = os.getenv("SERPER_API_KEY")
    if not api_key:
        return [{"error": "SERPER_API_KEY belum diset"}]
    payload = {"q": query, "gl": "id", "hl": "id", "num": n}
    headers = {"X-API-KEY": api_key, "Content-Type": "application/json"}
    for attempt in range(3):
        try:
            resp = requests.post("https://google.serper.dev/search",
                                 headers=headers, json=payload, timeout=20)
            resp.raise_for_status()
            hits = resp.json().get("organic", [])[:n]
            return [{"judul": h.get("title", ""), "url": h.get("link", ""),
                     "cuplikan": (h.get("snippet") or "")[:250]} for h in hits]
        except Exception as e:
            if attempt == 2:
                return [{"error": f"{type(e).__name__}: {str(e)[:120]}"}]
            time.sleep(2 * (attempt + 1))


def _tavily(query, n=4):
    from tavily import TavilyClient
    tv = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    try:
        res = tv.search(query, max_results=n, search_depth="basic")
        return [{"judul": h.get("title", ""), "url": h.get("url", ""),
                 "cuplikan": (h.get("content") or "")[:250]} for h in res.get("results", [])]
    except Exception as e:
        return [{"error": f"{type(e).__name__}: {str(e)[:120]}"}]


def cari_web(query, situs=None, backend="serper"):
    q = f"site:{situs} {query}" if situs else query
    k = _key(q, None)

    if k in _CACHE[backend]:
        SEARCH_COUNTER["cache_hit"] += 1
        return _CACHE[backend][k]

    hasil = _serper(q) if backend == "serper" else _tavily(q)
    SEARCH_COUNTER[backend] += 1

    _CACHE[backend][k] = hasil
    if SEARCH_COUNTER[backend] % 10 == 0:
        _save_cache(backend)
    return hasil


h = cari_web("laporan kinerja 2025", situs="kemenperin.go.id", backend=BACKEND)
if h and "error" in h[0]:
    print(f"[{BACKEND}] GAGAL: {h[0]['error']}")
else:
    print(f"[{BACKEND}] OK, {len(h)} hasil | contoh: {h[0]['judul'][:60]}")

[serper] OK, 4 hasil | contoh: [PDF] LAPORAN KINERJA TAHUN 2025


## 4. Prompt

In [4]:
SYSTEM_PROMPT = """Kamu verifikator realisasi indikator RPJMN 2025-2029 Indonesia.
Tentukan apakah PROGRAM/KEGIATAN pemerintah untuk indikator ini DILAKSANAKAN pada TAHUN 2025.

CARA MENCARI (wajib urut):
1. Pencarian PERTAMA harus TANPA parameter 'situs'. Contoh:
   "laporan kinerja 2025 <nama K/L> <kata kunci indikator>"
2. Gunakan 'situs' HANYA jika domain resminya sudah muncul di hasil pencarian sebelumnya.
   JANGAN menebak domain. Domain yang salah selalu mengembalikan 0 hasil.
3. Jika hasil kosong, ganti kata kunci. Jangan mengulang query yang sama.

Prioritas sumber: LKj/LAKIP 2025 K/L > siaran pers resmi K/L > Bappenas (e-Monev, RKP)
> BPS/Setkab > media nasional yang mengutip pejabat atau dokumen resmi.
TOLAK: blog, opini, analisis konsultan.

Label:
- true  = ada bukti KEGIATAN pemerintah terkait indikator ini BERJALAN pada 2025:
          pekerjaan fisik, kegiatan teknis, siaran pers kegiatan, atau capaian di LKj 2025.
          Angka capaian TIDAK wajib — berita kegiatan konkret bertanggal 2025 sudah cukup.
- false = yang ada hanya rencana, target, anggaran, rapat, atau kajian; ATAU buktinya
          dari tahun lain; ATAU hanya angka statistik tanpa bukti ada program
          pemerintah yang menggarap indikator ini pada 2025.

TAHUN: dokumen terbit 2026 boleh dipakai HANYA bila isinya melaporkan pelaksanaan 2025
(mis. LKj 2025 yang terbit awal 2026). Dokumen 2026 berisi rencana atau persiapan
BUKAN bukti pelaksanaan 2025.

Maksimal 3 kali cari_web. Bila tetap tidak ada bukti, jawab false. JANGAN mengarang URL.

implementation_summary WAJIB diawali persis "Program berjalan, ditandai dengan " atau
"Program belum berjalan, ditandai dengan ", lalu bukti konkret. Maksimal 3 kalimat.

Balas HANYA JSON, tanpa markdown:
{"realisasi": true, "implementation_summary": "Program berjalan, ditandai dengan ...", "source_url": "https://...", "confidence": 85, "source_type": "lkj", "evidence_year": 2025}
confidence: bilangan bulat 0-100, seberapa yakin label ini benar
source_type: lkj | siaran_pers | bappenas | bps_setkab | media | tidak_ada
evidence_year: 2025 | 2026 | lain | tidak_ada"""


def user_prompt(row):
    return (
        f"Sektor: {row['sektor']}\n"
        f"K/L: {row['kl']}\n"
        f"Indikator: {row['sub_indikator']}\n"
        f"Target 2025: {row['target_2025']} {row['satuan']} (baseline 2024: {row['baseline_2024']})\n"
        f"Kata kunci: {row['keywords']}"
    )


TOOLS = [{
    "type": "function",
    "function": {
        "name": "cari_web",
        "description": "Cari di web. Isi 'situs' untuk membatasi ke domain resmi, mis. kemenperin.go.id",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Kata kunci pencarian"},
                "situs": {"type": "string", "description": "Domain, opsional"},
            },
            "required": ["query"],
        },
    },
}]

print(SYSTEM_PROMPT)

Kamu verifikator realisasi indikator RPJMN 2025-2029 Indonesia.
Tentukan apakah PROGRAM/KEGIATAN pemerintah untuk indikator ini DILAKSANAKAN pada TAHUN 2025.

CARA MENCARI (wajib urut):
1. Pencarian PERTAMA harus TANPA parameter 'situs'. Contoh:
   "laporan kinerja 2025 <nama K/L> <kata kunci indikator>"
2. Gunakan 'situs' HANYA jika domain resminya sudah muncul di hasil pencarian sebelumnya.
   JANGAN menebak domain. Domain yang salah selalu mengembalikan 0 hasil.
3. Jika hasil kosong, ganti kata kunci. Jangan mengulang query yang sama.

Prioritas sumber: LKj/LAKIP 2025 K/L > siaran pers resmi K/L > Bappenas (e-Monev, RKP)
> BPS/Setkab > media nasional yang mengutip pejabat atau dokumen resmi.
TOLAK: blog, opini, analisis konsultan.

Label:
- true  = ada bukti KEGIATAN pemerintah terkait indikator ini BERJALAN pada 2025:
          pekerjaan fisik, kegiatan teknis, siaran pers kegiatan, atau capaian di LKj 2025.
          Angka capaian TIDAK wajib — berita kegiatan konkret bertanggal 2

## 5. Tool-calling loop

In [ ]:
def parse_json(txt):
    if not txt:
        return None
    t = re.sub(r"```(?:json)?", "", txt).strip()
    m = re.search(r"\{.*\}", t, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def _conf_int(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return max(0, min(100, int(v)))
    s = str(v).strip().lower()
    m = re.search(r"\d+", s)
    if m:
        return max(0, min(100, int(m.group())))
    return {"tinggi": 85, "sedang": 60, "rendah": 30}.get(s)


def verifikasi(row, model=None, backend=None, max_rounds=None,
               max_tokens=None, temperature=None):
    model       = model or MODEL
    backend     = backend or BACKEND
    max_rounds  = max_rounds if max_rounds is not None else MAX_ROUNDS
    max_tokens  = max_tokens if max_tokens is not None else MAX_TOKENS
    temperature = temperature if temperature is not None else TEMPERATURE

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(row)},
    ]
    seen_urls, queries, tool_calls_log = [], [], []
    n_search, n_search_gagal, in_tok, out_tok = 0, 0, 0, 0
    t0 = time.perf_counter()

    def _base(status, **kw):
        out = {"status": status, "error": "", "raw_response": "",
               "latency_s": round(time.perf_counter() - t0, 2),
               "n_search": n_search, "n_search_gagal": n_search_gagal,
               "in_tok": in_tok, "out_tok": out_tok,
               "queries": queries, "seen_urls": seen_urls,
               "tool_calls_log": tool_calls_log}
        out.update(kw)
        return out

    for _ in range(max_rounds):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, tools=TOOLS, tool_choice="auto",
                temperature=temperature, max_tokens=max_tokens, timeout=180,
            )
        except Exception as e:
            return _base("FAIL", error=f"{type(e).__name__}: {str(e)[:160]}")

        u = resp.usage
        in_tok  += getattr(u, "prompt_tokens", 0) or 0
        out_tok += getattr(u, "completion_tokens", 0) or 0
        msg = resp.choices[0].message

        if msg.tool_calls:
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [{"id": tc.id, "type": "function",
                                "function": {"name": tc.function.name,
                                             "arguments": tc.function.arguments}}
                               for tc in msg.tool_calls],
            })
            for tc in msg.tool_calls:
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except Exception:
                    args = {}
                q, situs = args.get("query", ""), args.get("situs")
                queries.append(f"{q}" + (f" [site:{situs}]" if situs else ""))
                hasil = cari_web(q, situs, backend=backend) if q else []
                n_search += 1
                if not hasil or (hasil and "error" in hasil[0]):
                    n_search_gagal += 1
                seen_urls += [h.get("url", "") for h in hasil if h.get("url")]
                tool_calls_log.append({"query": q, "situs": situs, "hasil": hasil})
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(hasil, ensure_ascii=False)[:3000]})
            continue

        txt = (msg.content or "").strip()
        if not txt:
            ak = msg.model_dump().get("provider_specific_fields") or {}
            txt = (ak.get("reasoning_content") or "").strip()

        data = parse_json(txt)
        if data is None:
            return _base("PARSE_FAIL", raw_response=txt[:4000])

        src = str(data.get("source_url") or "").strip()
        return _base(
            "OK",
            raw_response=txt[:4000],
            pred=bool(data.get("realisasi")),
            summary=str(data.get("implementation_summary") or "").strip(),
            source_url=src,
            confidence=_conf_int(data.get("confidence")),
            source_type=str(data.get("source_type") or "").strip().lower(),
            evidence_year=str(data.get("evidence_year") or "").strip().lower(),
            url_dari_search=bool(src) and any(src.rstrip("/") == u.rstrip("/") for u in seen_urls),
        )

    return _base("MAX_ROUNDS")

## 6. Uji satu baris

In [6]:
row = TARGET.iloc[0]
print("INDIKATOR:", str(row["sub_indikator"])[:90])
print("K/L      :", row["kl"])
print("GOLD     :", "TRUE" if row["_gold"] else "FALSE")
print("=" * 78)

h = verifikasi(row)

print("STATUS   :", h["status"], f"| {h['latency_s']}s | {h['n_search']} pencarian "
      f"({h['n_search_gagal']} gagal)")
print("TOKEN    :", f"in={h['in_tok']} out={h['out_tok']}")
print("\nQUERY:")
for q in h["queries"]:
    print("  -", q)

if h["status"] == "OK":
    print("\npred          :", h["pred"])
    print("confidence    :", h["confidence"], "(harus angka 0-100)")
    print("source_type   :", h["source_type"])
    print("evidence_year :", h["evidence_year"])
    print("url_dari_search:", h["url_dari_search"])
    print("source_url    :", h["source_url"][:100])
    print("\nsummary:", h["summary"])
    tmpl = h["summary"].startswith(("Program berjalan, ditandai dengan",
                                    "Program belum berjalan, ditandai dengan"))
    print("\npakai template:", tmpl)
else:
    print("\nDETAIL:", h.get("error") or h.get("raw_response", "")[:400])

INDIKATOR: KP 02.12.08 - Kabupaten/kota yang mendeklarasikan 5 Pilar STBM
K/L      : KEMENTERIAN KESEHATAN
GOLD     : TRUE
STATUS   : OK | 9.29s | 3 pencarian (2 gagal)
TOKEN    : in=6066 out=320

QUERY:
  - laporan kinerja 2025 KEMENTERIAN KESEHATAN STBM 5 pilar deklarasi
  - laporan kinerja 2025 KEMENTERIAN KESEHATAN deklarasi 5 pilar STBM [site:kemenkes.go.id]
  - siaran pers KEMENTERIAN KESEHATAN 2025 deklarasi 5 pilar STBM [site:kemenkes.go.id]

pred          : True
confidence    : 85 (harus angka 0-100)
source_type   : media
evidence_year : 2025
url_dari_search: True
source_url    : https://malangkota.go.id/2025/10/01/verifikator-stbm-5-pilar-kunjungi-kota-malang/

summary: Program berjalan, ditandai dengan deklarasi 5 Pilar STBM di Kota Malang pada 1 Oktober 2025 dan verifikasi di Kecamatan Prambanan, Klaten, serta kegiatan di Kabupaten Sleman yang melibatkan pemerintah daerah dan puskesmas.

pakai template: True


## 7. Run 292 baris

In [7]:
def load_done(path):
    done = {}
    if path.exists():
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    done[rec["row_no"]] = rec
                except Exception:
                    continue
    return done


done = load_done(F_JSONL)
sisa = [r for _, r in TARGET.iterrows() if r["no"] not in done]

print(f"Sudah selesai : {len(done)}")
print(f"Sisa          : {len(sisa)}")
print(f"Model         : {MODEL}")
print(f"Backend       : {BACKEND} | temperature={TEMPERATURE}\n")

t_start = time.perf_counter()
for i, r in enumerate(sisa, 1):
    h = verifikasi(r)

    rec = {
        "run_id": RUN_TS, "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": MODEL, "backend": BACKEND, "temperature": TEMPERATURE,
        "row_no": int(r["no"]), "id": r["id"],
        "sektor": r["sektor"], "kl": r["kl"],
        "sub_indikator": r["sub_indikator"],
        "target_2025": str(r["target_2025"]), "satuan": r["satuan"],
        "gold": bool(r["_gold"]), "gold_suspect": bool(r["_gold_suspect"]),
        "summary_gold": r["implementation_summary"],
        "status": h["status"], "error": h.get("error", ""),
        "pred": h.get("pred"), "confidence": h.get("confidence"),
        "source_type": h.get("source_type", ""), "evidence_year": h.get("evidence_year", ""),
        "summary_pred": h.get("summary", ""), "source_url_pred": h.get("source_url", ""),
        "url_dari_search": h.get("url_dari_search"),
        "n_search": h["n_search"], "n_search_gagal": h["n_search_gagal"],
        "in_tok": h["in_tok"], "out_tok": h["out_tok"], "latency_s": h["latency_s"],
        "queries": h["queries"],
        "tool_calls_log": h["tool_calls_log"],
        "raw_response": h.get("raw_response", ""),
    }
    with open(F_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    ok = h["status"] == "OK"
    benar = (h.get("pred") == bool(r["_gold"])) if ok else None
    mark = ("BENAR" if benar else "SALAH") if ok else h["status"]
    sisa_menit = (time.perf_counter() - t_start) / i * (len(sisa) - i) / 60
    print(f"  [{i:3d}/{len(sisa)}] no={r['no']:<4} gold={'T' if r['_gold'] else 'F'} "
          f"pred={'T' if h.get('pred') else 'F' if ok else '-'} {mark:10s} "
          f"conf={str(h.get('confidence','-')):>4} {h['n_search']}x "
          f"{h['latency_s']:>6.1f}s  ~{sisa_menit:.0f}m lagi")

_save_cache(BACKEND)
print(f"\nSelesai. Pemakaian pencarian: {SEARCH_COUNTER}")

Sudah selesai : 0
Sisa          : 292
Model         : watsonx-qwen3-30b-a3b-instruct-2507
Backend       : serper | temperature=0.0

  [  1/292] no=1    gold=T pred=T BENAR      conf=  85 3x    0.5s  ~3m lagi
  [  2/292] no=2    gold=T pred=F SALAH      conf=  90 3x   20.2s  ~50m lagi
  [  3/292] no=3    gold=T pred=T BENAR      conf=  85 3x    8.2s  ~46m lagi
  [  4/292] no=4    gold=T pred=T BENAR      conf=  85 3x    9.0s  ~45m lagi
  [  5/292] no=5    gold=T pred=F SALAH      conf=  70 3x    5.8s  ~42m lagi
  [  6/292] no=6    gold=T pred=T BENAR      conf=  90 3x    9.2s  ~42m lagi
  [  7/292] no=7    gold=T pred=F SALAH      conf=  90 3x    5.0s  ~39m lagi
  [  8/292] no=8    gold=T pred=T BENAR      conf=  90 3x   16.0s  ~44m lagi
  [  9/292] no=9    gold=T pred=T BENAR      conf=  90 3x   10.0s  ~44m lagi
  [ 10/292] no=10   gold=T pred=F SALAH      conf=  90 3x   17.3s  ~48m lagi
  [ 11/292] no=11   gold=F pred=T SALAH      conf=  95 3x    7.3s  ~46m lagi
  [ 12/292] no=12   go

## 8. Tulis CSV

In [8]:
done = load_done(F_JSONL)
detail = pd.DataFrame(list(done.values()))

detail["benar"] = detail.apply(
    lambda r: (r["pred"] == r["gold"]) if r["status"] == "OK" else None, axis=1)
detail["pakai_template"] = detail["summary_pred"].fillna("").str.startswith(
    ("Program berjalan, ditandai dengan", "Program belum berjalan, ditandai dengan"))
detail["domain"] = detail["source_url_pred"].fillna("").str.extract(r"https?://([^/]+)")[0]
detail["domain_goid"] = detail["domain"].fillna("").str.contains(r"\.go\.id$", regex=True)
detail["n_query"] = detail["queries"].apply(lambda x: len(x) if isinstance(x, list) else 0)
detail["queries_str"] = detail["queries"].apply(
    lambda x: " | ".join(x) if isinstance(x, list) else "")

KOLOM_CSV = [
    "run_id", "timestamp", "model", "backend", "temperature",
    "row_no", "id", "sektor", "kl", "sub_indikator", "target_2025", "satuan",
    "gold", "gold_suspect", "pred", "benar", "confidence",
    "source_type", "evidence_year",
    "summary_pred", "summary_gold", "source_url_pred",
    "domain", "domain_goid", "url_dari_search", "pakai_template",
    "status", "error", "n_search", "n_search_gagal", "n_query", "queries_str",
    "in_tok", "out_tok", "latency_s",
]
detail[KOLOM_CSV].to_csv(F_DETAIL, index=False, encoding="utf-8-sig")

print("Tersimpan:")
print(" ", F_DETAIL, f"({len(detail)} baris)")
print(" ", F_JSONL, f"({F_JSONL.stat().st_size/1e6:.1f} MB, data mentah)")

Tersimpan:
  ..\data\output\detail_watsonx-qwen3-30b-a3b-instruct-2_serper_20260917_150324.csv (292 baris)
  ..\data\output\raw_watsonx-qwen3-30b-a3b-instruct-2_serper.jsonl (1.8 MB, data mentah)


## 9. Pemeriksaan kelayakan

Memastikan datanya layak diserahkan: tidak banyak yang gagal, field baru terisi, tidak ada halusinasi URL yang mencolok.

In [9]:
ok = detail[detail["status"] == "OK"]

print("=== KELENGKAPAN ===")
print(detail["status"].value_counts().to_string())
print(f"\nField terisi (dari {len(ok)} OK):")
print(f"  confidence numerik : {ok['confidence'].notna().sum()}")
print(f"  source_type        : {(ok['source_type'].fillna('') != '').sum()}")
print(f"  evidence_year      : {(ok['evidence_year'].fillna('') != '').sum()}")
print(f"  pakai template     : {ok['pakai_template'].sum()}")

print("\n=== INTEGRITAS SUMBER ===")
print(f"  url_dari_search True : {ok['url_dari_search'].sum()} / {len(ok)}")
print(f"  domain .go.id        : {ok['domain_goid'].sum()} / {len(ok)}")
print(f"  pencarian gagal      : {detail['n_search_gagal'].sum()} dari {detail['n_search'].sum()}")

print("\n=== SEBARAN (bahan tim analisis) ===")
print("\nsource_type:")
print(ok["source_type"].value_counts().to_string())
print("\nevidence_year:")
print(ok["evidence_year"].value_counts().to_string())
print("\nconfidence:")
print(ok["confidence"].describe().to_string())

print("\n=== SKOR KASAR ===")
n_true, n_false = ok["gold"].sum(), (~ok["gold"]).sum()
tp = ((ok["gold"]) & (ok["pred"])).sum(); tn = ((~ok["gold"]) & (~ok["pred"])).sum()
print(f"  akurasi      : {(tp+tn)/len(ok):.3f}   (baseline asal TRUE = {n_true/len(ok):.3f})")
print(f"  recall_TRUE  : {tp/n_true:.3f}")
print(f"  recall_FALSE : {tn/n_false:.3f}   (n={n_false}, presisi rendah)")
print(f"\n  token total  : in={detail['in_tok'].sum():,} out={detail['out_tok'].sum():,}")
print(f"  pencarian    : {detail['n_search'].sum():,}")

if (detail["status"] != "OK").sum() > 0:
    print("\n=== BARIS GAGAL (pertimbangkan jalankan ulang) ===")
    for _, r in detail[detail["status"] != "OK"].iterrows():
        print(f"  no={r['row_no']:<4} {r['status']:12s} {str(r['error'])[:80]}")

=== KELENGKAPAN ===
status
OK    292

Field terisi (dari 292 OK):
  confidence numerik : 292
  source_type        : 292
  evidence_year      : 292
  pakai template     : 292

=== INTEGRITAS SUMBER ===
  url_dari_search True : 234 / 292
  domain .go.id        : 235 / 292
  pencarian gagal      : 90 dari 876

=== SEBARAN (bahan tim analisis) ===

source_type:
source_type
siaran_pers    88
lkj            84
tidak_ada      59
media          32
bappenas       22
bps_setkab      6
renstra         1

evidence_year:
evidence_year
2025         229
tidak_ada     56
2026           5
2024           1
2023           1

confidence:
count    292.000000
mean      89.469178
std        3.312334
min       70.000000
25%       90.000000
50%       90.000000
75%       90.000000
max       95.000000

=== SKOR KASAR ===
  akurasi      : 0.592   (baseline asal TRUE = 0.979)
  recall_TRUE  : 0.601
  recall_FALSE : 0.167   (n=6, presisi rendah)

  token total  : in=2,175,232 out=100,635
  pencarian    : 876
